# 農地問題エリア検出システム

**画像ソース**: 国土地理院 全国最新写真（シームレス） - APIキー不要・完全無料

## 実行順序
1. セル1〜3: 初期セットアップ（最初に一度だけ）
2. セル4: 設定（GeoJSONパスを変更）
3. セル5〜9: 画像取得
4. **手動作業**: `data/unlabeled/` の画像を `data/farmland/` と `data/problem/` に振り分け
5. セル10〜13: CNN学習
6. セル14〜16: 推論・自動仕分け
7. **手動作業**: `data/review/` の画像を振り分け
8. セル17: 継続学習（精度が上がるまで 手動仕分け→継続学習 を繰り返す）

In [ ]:
# ===== セル1: ライブラリインストール（初回のみ） =====
!pip install geopandas shapely Pillow torch torchvision tqdm matplotlib requests scikit-learn pandas pyproj ipywidgets mercantile -q
# tqdmのプログレスバーをJupyterで表示するために ipywidgets が必要
# インストール後にカーネルを再起動してください（メニュー: Kernel → Restart）

In [ ]:
# ===== セル2: インポート =====
import hashlib
import io
import math
import shutil
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models

# tqdm: notebook環境で固まる場合は tqdm.auto が自動判別して安全
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {DEVICE}')

In [ ]:
# ===== セル3: ディレクトリ作成 =====
for d in ['data/unlabeled', 'data/farmland', 'data/problem', 'data/review', 'models', 'logs']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('ディレクトリ作成完了')

In [ ]:
# ===== セル4: 設定（ここを自分の環境に合わせて変更） =====
GEOJSON_PATH = 'data/farmland.geojson'  # ← 農地GeoJSONのパスを指定
OUTPUT_DIR   = 'data/unlabeled'
MAX_POLYGONS = 200    # None で全件。まず小さい数でテスト推奨
ZOOM         = 18     # 18=約0.6m/pixel。大きいポリゴンは自動で下げる
OUT_SIZE     = 512    # 保存する画像サイズ（px）
MARGIN_TILES = 1      # ポリゴン周囲の余白タイル数

# 国土地理院タイルURL（APIキー不要・無料）
GSI_TILE_URL = 'https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg'

In [ ]:
# ===== セル5: 画像取得ヘルパー関数 =====
import math
import re
import mercantile

TILE_SIZE = 256  # GSIタイルは256x256px固定

# ---------- ファイル名生成（所在・地番・地目） ----------
SHOZAI_COL   = 'Address'                        # 所在のカラム名
CHIBAN_COL   = 'Tiban'                          # 地番のカラム名
LANDTYPE_COL = 'ClassificationOfLandCodeName'  # 地目のカラム名（田・畑など）

def _sanitize_filename(s):
    s = str(s).strip()
    s = s.replace('/', '-').replace('\\', '-').replace(':', '-')
    s = re.sub(r'[<>"|?*\x00-\x1f]', '', s)
    s = s.strip('. ')
    return s[:80]

def polygon_filename(row, idx, geom):
    """
    地目_所在_地番 の形式でファイル名を生成。
    カラムがない・空の場合は MD5 uid にフォールバック。
    重複時の枝番付与は行わない。
    """
    shozai   = row.get(SHOZAI_COL)   if SHOZAI_COL   in row.index else None
    chiban   = row.get(CHIBAN_COL)   if CHIBAN_COL   in row.index else None
    landtype = row.get(LANDTYPE_COL) if LANDTYPE_COL in row.index else None

    has_shozai   = shozai   is not None and pd.notna(shozai)   and str(shozai).strip()
    has_chiban   = chiban   is not None and pd.notna(chiban)   and str(chiban).strip()
    has_landtype = landtype is not None and pd.notna(landtype) and str(landtype).strip()

    if has_shozai and has_chiban:
        base = _sanitize_filename(f'{shozai}_{chiban}')
    elif has_shozai:
        base = _sanitize_filename(f'{shozai}_{idx}')
    else:
        base = hashlib.md5(f'{idx}_{geom.wkt[:200]}'.encode()).hexdigest()[:12]

    if has_landtype:
        prefix = _sanitize_filename(str(landtype).strip())
        return f'{prefix}_{base}'
    return base

def polygon_uid(geom, idx):
    return hashlib.md5(f'{idx}_{geom.wkt[:200]}'.encode()).hexdigest()[:12]

# ---------- タイル座標・ピクセル変換 ----------

def lonlat_to_global_pixel(lon, lat, zoom):
    lat = max(min(lat, 85.05112878), -85.05112878)
    siny = math.sin(math.radians(lat))
    scale = TILE_SIZE * (2 ** zoom)
    gx = (lon + 180.0) / 360.0 * scale
    gy = (0.5 - math.log((1 + siny) / (1 - siny)) / (4 * math.pi)) * scale
    return gx, gy

def lonlat_to_mosaic_pixel(lon, lat, zoom, left_tile_x, top_tile_y):
    gx, gy = lonlat_to_global_pixel(lon, lat, zoom)
    return gx - left_tile_x * TILE_SIZE, gy - top_tile_y * TILE_SIZE

def latlon_to_tile_xy(lat, lon, zoom):
    t = mercantile.tile(lon, lat, zoom)
    return t.x, t.y

# ---------- タイル取得 ----------

def fetch_tile(tx, ty, zoom, session, retries=3):
    url = GSI_TILE_URL.format(z=zoom, x=tx, y=ty)
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code != 200:
                return Image.new('RGB', (TILE_SIZE, TILE_SIZE), (255, 255, 255))
            return Image.open(io.BytesIO(resp.content)).convert('RGB')
        except Exception:
            if attempt < retries - 1:
                time.sleep(1.5 ** attempt)
    return Image.new('RGB', (TILE_SIZE, TILE_SIZE), (255, 255, 255))

# ---------- モザイク合成 ----------

def fetch_region_image(bounds, zoom, margin=1, session=None):
    minx, miny, maxx, maxy = bounds
    if session is None:
        session = requests.Session()
    tiles = list(mercantile.tiles(minx, miny, maxx, maxy, [zoom]))
    if not tiles:
        raise ValueError("No tiles found for bounds")

    xs = [t.x for t in tiles]; ys = [t.y for t in tiles]
    tx_min, tx_max = min(xs) - margin, max(xs) + margin
    ty_min, ty_max = min(ys) - margin, max(ys) + margin

    n_cols = tx_max - tx_min + 1
    n_rows = ty_max - ty_min + 1
    canvas = Image.new('RGB', (n_cols * TILE_SIZE, n_rows * TILE_SIZE), (255, 255, 255))

    for ty in range(ty_min, ty_max + 1):
        for tx in range(tx_min, tx_max + 1):
            tile = fetch_tile(tx, ty, zoom, session)
            canvas.paste(tile, ((tx - tx_min) * TILE_SIZE, (ty - ty_min) * TILE_SIZE))

    left,  top    = lonlat_to_mosaic_pixel(minx, maxy, zoom, tx_min, ty_min)
    right, bottom = lonlat_to_mosaic_pixel(maxx, miny, zoom, tx_min, ty_min)
    pad = 20
    crop_box = (max(0, int(left) - pad),  max(0, int(top) - pad),
                min(canvas.width, int(right) + pad), min(canvas.height, int(bottom) + pad))
    return canvas.crop(crop_box), tx_min, ty_min, crop_box

# ---------- ポリゴン輪郭描画 ----------

def draw_polygon_outline(img, geom, tx_min, ty_min, crop_box, zoom,
                         color=(255, 0, 0), width=3):
    draw = ImageDraw.Draw(img)
    ox, oy = crop_box[0], crop_box[1]

    def geom_to_pixels(polygon):
        coords = list(polygon.exterior.coords)
        pts = []
        for lon, lat in coords:
            gx, gy = lonlat_to_global_pixel(lon, lat, zoom)
            px = gx - tx_min * TILE_SIZE - ox
            py = gy - ty_min * TILE_SIZE - oy
            pts.append((px, py))
        return pts

    if geom.geom_type == 'Polygon':
        pts = geom_to_pixels(geom)
        if len(pts) >= 2:
            draw.line(pts + [pts[0]], fill=color, width=width)
    elif geom.geom_type == 'MultiPolygon':
        for poly in geom.geoms:
            pts = geom_to_pixels(poly)
            if len(pts) >= 2:
                draw.line(pts + [pts[0]], fill=color, width=width)
    return img


In [ ]:
# ===== セル6: 接続テスト（タイル1枚だけ取得・数秒で完了） =====
# 広い範囲をzoom=18で取得すると数百枚になるため、1枚だけで確認する
session = requests.Session()
session.headers.update({'User-Agent': 'farmland-detector/1.0 (research)'})

test_lat, test_lon = 36.10, 140.08  # つくば市付近
tx, ty = latlon_to_tile_xy(test_lat, test_lon, zoom=18)
test_tile = fetch_tile(tx, ty, zoom=18, session=session)

plt.figure(figsize=(5, 5))
plt.imshow(test_tile)
plt.title(f'接続テスト OK（zoom=18, 1タイル=256x256px）\n国土地理院 全国最新写真')
plt.axis('off')
plt.show()
print(f'接続OK - タイル座標: z=18, x={tx}, y={ty}')
print('※ 実際の農地画像取得（セル8）ではポリゴン範囲の複数タイルを自動結合します')

In [ ]:
# ===== セル7: GeoJSON読み込み・分布確認 =====
gdf = gpd.read_file(GEOJSON_PATH)
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)

# 農業振興地域内・農用地区域（青地）かつ対象地目のみに絞り込む
TARGET_NOUSHIN   = '農業振興地域内・農用地区域内(青地)'
TARGET_LANDTYPES = ['田', '畑', '樹園地', '採草放牧地']
if 'SectionOfNoushinhouCodeName' in gdf.columns and 'ClassificationOfLandCodeName' in gdf.columns:
    gdf = gdf[
        (gdf['SectionOfNoushinhouCodeName'] == TARGET_NOUSHIN) &
        (gdf['ClassificationOfLandCodeName'].isin(TARGET_LANDTYPES))
    ].reset_index(drop=True)
    print(f'フィルター後ポリゴン数: {len(gdf)}')
else:
    print('警告: SectionOfNoushinhouCodeName または ClassificationOfLandCodeName カラムが見つかりません。フィルターをスキップします。')

# 面積30㎡未満を除外
if 'AreaOnRegistry' in gdf.columns:
    gdf = gdf[pd.to_numeric(gdf['AreaOnRegistry'], errors='coerce').fillna(0) >= 30].reset_index(drop=True)
    print(f'面積フィルター後: {len(gdf)}件')

if MAX_POLYGONS:
    gdf = gdf.head(MAX_POLYGONS)

print(f'処理対象ポリゴン数: {len(gdf)}')
gdf.plot(figsize=(10, 8), color='green', alpha=0.3, edgecolor='black', linewidth=0.3)
plt.title('農地ポリゴン分布')
plt.show()


In [ ]:
# ===== セル8: 画像取得メインループ（並列化 + 小ポリゴン対応版） =====
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from tqdm import tqdm as tqdm_cli

WORKERS  = 4      # 同時並列数。GSIサーバー負荷軽減のため4以上に上げないこと
MIN_SPAN = 0.0003 # これ未満のポリゴンは中心から拡張して取得（約30m）

# ---------- ヘルパー ----------

def calc_auto_zoom(bounds, max_zoom=18, max_tiles=16):
    """ポリゴン範囲に対して適切なズームレベルを自動計算する。"""
    import mercantile as _mc
    for z in range(max_zoom, 10, -1):
        tiles = list(_mc.tiles(bounds[0], bounds[1], bounds[2], bounds[3], [z]))
        if len(tiles) <= max_tiles:
            return z
    return 11

def make_session():
    s = requests.Session()
    s.headers.update({'User-Agent': 'farmland-detector/1.0 (research)'})
    adapter = HTTPAdapter(pool_connections=WORKERS * 2, pool_maxsize=WORKERS * 4)
    s.mount('https://', adapter)
    return s

def ensure_min_bounds(bounds, min_span=MIN_SPAN):
    """ポリゴンが小さすぎる場合、中心から min_span に広げた bounds を返す。"""
    minx, miny, maxx, maxy = bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    if max(maxx - minx, maxy - miny) < min_span:
        half = min_span / 2
        return (cx - half, cy - half, cx + half, cy + half)
    return bounds

def fetch_region_image_parallel(bounds, zoom, margin=1, session=None):
    """mercantile でタイルリストを作り、並列取得してキャンバスに結合・クロップ。"""
    minx, miny, maxx, maxy = bounds
    if session is None:
        session = make_session()

    tiles = list(mercantile.tiles(minx, miny, maxx, maxy, [zoom]))
    if not tiles:
        raise ValueError("No tiles found for bounds")

    xs = [t.x for t in tiles]; ys = [t.y for t in tiles]
    tx_min, tx_max = min(xs) - margin, max(xs) + margin
    ty_min, ty_max = min(ys) - margin, max(ys) + margin

    n_cols = tx_max - tx_min + 1
    n_rows = ty_max - ty_min + 1
    canvas = Image.new('RGB', (n_cols * TILE_SIZE, n_rows * TILE_SIZE), (255, 255, 255))

    all_coords = [(tx, ty) for ty in range(ty_min, ty_max + 1)
                            for tx in range(tx_min, tx_max + 1)]

    with ThreadPoolExecutor(max_workers=min(len(all_coords), WORKERS * 2)) as ex:
        future_to_pos = {ex.submit(fetch_tile, tx, ty, zoom, session): (tx, ty)
                         for tx, ty in all_coords}
        for fut in as_completed(future_to_pos):
            tx, ty = future_to_pos[fut]
            tile = fut.result()
            canvas.paste(tile, ((tx - tx_min) * TILE_SIZE, (ty - ty_min) * TILE_SIZE))

    left,  top    = lonlat_to_mosaic_pixel(minx, maxy, zoom, tx_min, ty_min)
    right, bottom = lonlat_to_mosaic_pixel(maxx, miny, zoom, tx_min, ty_min)
    pad = 20
    crop_box = (max(0, int(left) - pad),  max(0, int(top) - pad),
                min(canvas.width, int(right) + pad), min(canvas.height, int(bottom) + pad))
    return canvas.crop(crop_box), tx_min, ty_min, crop_box

# ---------- 1ポリゴン処理 ----------

_write_lock = threading.Lock()

def process_polygon(idx, row, output_dir, session):
    geom = row.geometry

    # ファイル名を所在・地番から生成（重複時は _2, _3 ... を付与）
    with _write_lock:
        fname = polygon_filename(row, idx, geom)
    out_path = output_dir / f'{fname}.jpg'

    if out_path.exists():
        return 'exists', None

    bounds = geom.bounds
    span   = max(bounds[2] - bounds[0], bounds[3] - bounds[1])

    fetch_bounds = ensure_min_bounds(bounds)
    zoom = calc_auto_zoom(fetch_bounds, max_zoom=ZOOM)

    try:
        img, tx0, ty0, cbox = fetch_region_image_parallel(
            fetch_bounds, zoom=zoom, margin=MARGIN_TILES, session=session)
        img = draw_polygon_outline(img, geom, tx0, ty0, cbox, zoom)
        img = img.resize((OUT_SIZE, OUT_SIZE), Image.LANCZOS)
        img.save(out_path, 'JPEG', quality=92)
        return 'success', {'filename': fname, 'path': str(out_path), 'zoom': zoom,
                           'span': round(span, 6), 'expanded': span < MIN_SPAN}
    except Exception as e:
        return 'error', str(e)

# ---------- メインループ ----------

output_dir   = Path(OUTPUT_DIR)
session_pool = make_session()

success = skip_exists = error = 0
meta_records = []

pbar = tqdm_cli(total=len(gdf), desc='画像取得', unit='件', dynamic_ncols=True, leave=True)

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {
        executor.submit(process_polygon, idx, row, output_dir, session_pool): idx
        for idx, row in gdf.iterrows()
    }
    for fut in as_completed(futures):
        idx = futures[fut]
        status, data = fut.result()
        if status == 'success':
            success += 1
            with _write_lock:
                meta_records.append(data)
        elif status == 'exists':
            skip_exists += 1
        else:
            error += 1
        pbar.update(1)
        pbar.set_postfix(成功=success, 既存=skip_exists, エラー=error)

pbar.close()

# metadata.csv に追記（既存データとマージ）
meta_path = output_dir / 'metadata.csv'
if meta_path.exists() and meta_records:
    existing = pd.read_csv(meta_path)
    merged = pd.concat([existing, pd.DataFrame(meta_records)]).drop_duplicates('filename')
    merged.to_csv(meta_path, index=False)
elif meta_records:
    pd.DataFrame(meta_records).to_csv(meta_path, index=False)

print(f'\n--- 完了 ---')
print(f'  成功          : {success} 件')
print(f'  既存スキップ   : {skip_exists} 件')
print(f'  エラー        : {error} 件')
print(f'  合計保存済み   : {len(list(output_dir.glob("*.jpg")))} 枚')
if meta_records:
    print(f'\n【ファイル名サンプル（先頭5件）】')
    for r in meta_records[:5]:
        print(f'  {r["filename"]}.jpg')


In [ ]:
# ===== セル8b: エラー画像の再取得（fd上限修正 + 正しい成否判定） =====
# セル8でエラーになったidxを ERROR_LOG に貼り付けて実行する
import re, resource
from pathlib import Path
from tqdm.notebook import tqdm

# fd上限を引き上げ
soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
new_limit = min(hard, 65536)
resource.setrlimit(resource.RLIMIT_NOFILE, (new_limit, hard))
print(f'fd上限: {soft} → {new_limit}')

# ▼ セル8のエラーログをここに貼り付け（idx=NNNNN の行が含まれていればOK）
ERROR_LOG = """
[error] idx=12345: [Errno 24] Too many open files
"""  # ← ここを実際のエラーログに置き換える

error_idxs = sorted(set(int(m.group(1)) for m in re.finditer(r'idx=(\d+)', ERROR_LOG)))
print(f'再取得対象: {len(error_idxs)}件')

output_dir = Path(OUTPUT_DIR)
session    = make_session()

# ファイルが存在する場合（部分書き込みの可能性）は削除してから再取得
deleted = 0
for idx in error_idxs:
    row  = gdf.iloc[idx]
    geom = row.geometry
    fname = polygon_filename(row, idx, geom)
    p = output_dir / f'{fname}.jpg'
    if p.exists():
        p.unlink()
        deleted += 1
print(f'既存ファイル削除: {deleted}件（空/破損ファイルの可能性）')

# 再取得（process_polygon は 'success' / 'exists' / 'toobig' / 'error' を返す）
ok = err = existed = 0
for idx in tqdm(error_idxs, desc='再取得'):
    row   = gdf.iloc[idx]
    result, info = process_polygon(idx, row, output_dir, session)
    if result == 'success':
        ok += 1
    elif result == 'exists':
        existed += 1  # 削除しなかったファイルが存在する場合
    else:
        err += 1
        print(f'  [error] idx={idx}: {info}')

print(f'\n完了: 成功={ok}, 既存={existed}, エラー={err}')


In [ ]:
# ===== セル9: 取得画像のサムネイル確認（先頭12枚） =====
images  = sorted(output_dir.glob('*.jpg'))[:12]
meta_df = pd.read_csv(output_dir / 'metadata.csv') if (output_dir / 'metadata.csv').exists() else pd.DataFrame()
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axes.flat, images):
    ax.imshow(Image.open(p))
    row = meta_df[meta_df.uid == p.stem] if len(meta_df) else pd.DataFrame()
    z = int(row.zoom.values[0]) if len(row) else '?'
    ax.set_title(f'{p.stem[:8]}\nzoom={z}', fontsize=7)
    ax.axis('off')
for ax in axes.flat[len(images):]:
    ax.axis('off')
plt.suptitle('取得した農地画像（先頭12枚） - 国土地理院 全国最新写真')
plt.tight_layout()
plt.show()

In [ ]:
# ===== セル10: ラベリング状況確認 =====
# 画像取得後、data/unlabeled/ の画像を目視で確認し
# data/farmland/ → 正常な農地
# data/problem/  → 建物・道路が1/3以上の問題エリア
# に手動で振り分けてからこのセルを実行してください（各クラス50枚以上推奨）

for cls in ['farmland', 'problem', 'unlabeled', 'review']:
    n = len(list(Path(f'data/{cls}').glob('*.jpg'))) if Path(f'data/{cls}').exists() else 0
    print(f'  {cls:12s}: {n} 枚')

In [ ]:
# ===== セル11: モデル・学習関数の定義（EfficientNet-B2） =====
REGION     = 'Tokyo'   # ← 都道府県名（Tokyo / Kanagawa / Niigata など）
MODEL_VER  = 'v0'      # ← モデルのバージョン

DATA_DIR   = f'/Users/nk19187/Downloads/datafarm{REGION.lower()}'
MODEL_OUT  = f'models/model_{REGION.lower()}{MODEL_VER}.pth'
EPOCHS     = 50
BATCH_SIZE = 64
LR         = 1e-4
VAL_RATIO  = 0.15
PATIENCE   = 10  # early stopping

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'対象地域: {REGION}')
print(f'データ  : {DATA_DIR}')
print(f'保存先  : {MODEL_OUT}')
print(f'デバイス: {DEVICE}')

def build_transforms(train=True):
    if train:
        return T.Compose([
            T.Resize((260, 260)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.RandomRotation(15),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return T.Compose([
        T.Resize((260, 260)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

def build_model(num_classes=2):
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

print('EfficientNet-B2 モデル関数定義完了（入力260×260）')


In [ ]:
# ===== セル12: 初回学習 =====
# ▼ 事前に farmland_list / problem_list を定義しておくこと
#   ファイルは unlabeled/ にあってもOK（自動で探す）

import random
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image

PATIENCE = 10  # early stopping

TRAIN_CLASSES = sorted(['farmland', 'problem'])
class_to_idx  = {cls: i for i, cls in enumerate(TRAIN_CLASSES)}

# 探索するフォルダ（優先順）
_search_dirs = [
    Path(DATA_DIR) / 'farmland',
    Path(DATA_DIR) / 'problem',
    Path(DATA_DIR) / 'unlabeled',
    Path(DATA_DIR) / 'review',
]

def _resolve_path(p):
    p = Path(p)
    # 1. そのままで存在する
    if p.exists():
        return str(p.resolve())
    # 2. 各フォルダ以下でファイル名一致を探す
    for d in _search_dirs:
        candidate = d / p.name
        if candidate.exists():
            return str(candidate.resolve())
    # 3. 見つからなければそのまま返す（存在確認で除外される）
    return str(p)

samples = (
    [(_resolve_path(p), class_to_idx['farmland']) for p in farmland_list] +
    [(_resolve_path(p), class_to_idx['problem'])  for p in problem_list]
)

# 存在確認
missing = [p for p, _ in samples if not Path(p).exists()]
if missing:
    print(f'⚠️ 見つからなかったファイル: {len(missing)}件')
    for m in missing[:5]:
        print(f'  {m}')
    samples = [(p, lbl) for p, lbl in samples if Path(p).exists()]

random.seed(42)
random.shuffle(samples)

counts = [0, 0]
for _, lbl in samples:
    counts[lbl] += 1

print(f'farmland_list: {len(farmland_list)}件 → 有効: {counts[class_to_idx["farmland"]]}枚')
print(f'problem_list : {len(problem_list)}件 → 有効: {counts[class_to_idx["problem"]]}枚')

if 0 in counts:
    missing_cls = [TRAIN_CLASSES[i] for i, c in enumerate(counts) if c == 0]
    raise ValueError(f'⚠️ {missing_cls} に画像がありません')

# --- SimpleImageDataset ---
class SimpleImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# --- train/val 分割 ---
n_val   = max(1, int(len(samples) * VAL_RATIO))
n_train = len(samples) - n_val
train_ds = SimpleImageDataset(samples[:n_train], transform=build_transforms(train=True))
val_ds   = SimpleImageDataset(samples[n_train:], transform=build_transforms(train=False))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f'train={n_train}枚, val={n_val}枚')

# --- クラス重み・モデル・最適化 ---
class_weights = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
model     = build_model(num_classes=2).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history, best_val_acc, no_improve = [], 0.0, 0
Path(MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(1, EPOCHS + 1), desc='学習'):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)
    scheduler.step()
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    tqdm.write(f'Epoch {epoch:03d} | train={train_acc:.4f} loss={train_loss:.4f} | val={val_acc:.4f} loss={val_loss:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve   = 0
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'class_to_idx': class_to_idx, 'val_acc': val_acc}, MODEL_OUT)
        tqdm.write(f'  → モデル保存 (val_acc={val_acc:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            tqdm.write(f'Early stopping (patience={PATIENCE})')
            break

print(f'\n学習完了。最良 val_acc={best_val_acc:.4f}')

In [ ]:
# ===== セル13: 学習曲線グラフ =====
df_hist = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df_hist.epoch, df_hist.train_loss, label='train')
ax1.plot(df_hist.epoch, df_hist.val_loss,   label='val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(df_hist.epoch, df_hist.train_acc, label='train')
ax2.plot(df_hist.epoch, df_hist.val_acc,   label='val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('logs/training_curve.png', dpi=150)
plt.show()

In [ ]:
# ===== セル14: 推論設定 =====
PREDICT_MODEL   = MODEL_OUT                      # セル11の REGION 設定を継承
PREDICT_INPUT   = Path(DATA_DIR) / 'unlabeled'   # 推論対象フォルダ
THRESHOLD_PROB  = 0.85   # problem判定の最低確信度（これ未満はreviewへ）
THRESHOLD_FARM  = 0.5    # farmland判定の最低確信度
BATCH_SIZE_INFER = 64    # B2 + MPS: 64推奨
DRY_RUN         = True   # True=移動しない（確認用）。False で実際に移動

print(f'推論モデル  : {PREDICT_MODEL}')
print(f'推論対象    : {PREDICT_INPUT}')
print(f'THRESHOLD_PROB(problem) : {THRESHOLD_PROB}')
print(f'THRESHOLD_FARM(farmland): {THRESHOLD_FARM}')
print(f'DRY_RUN     : {DRY_RUN}')


In [ ]:
# ===== セル15: 推論実行（バッチ推論・高速版） =====
from torch.utils.data import Dataset, DataLoader

def _build_model_for_infer(num_classes=2):
    """チェックポイントのアーキテクチャを自動判別してモデルを返す。"""
    ckpt = torch.load(PREDICT_MODEL, map_location='cpu')
    # features.1.1 の有無でB0/B2を判別（B2のみfeatures.1に2ブロック存在）
    if 'features.1.1.block.0.0.weight' in ckpt['model']:
        m = models.efficientnet_b2(weights=None)
        print('アーキテクチャ自動判別: B2')
    else:
        m = models.efficientnet_b0(weights=None)
        print('アーキテクチャ自動判別: B0')
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m, ckpt

pred_model, state = _build_model_for_infer()
pred_model = pred_model.to(DEVICE)
pred_model.load_state_dict(state['model'])
pred_model.eval()
idx_to_class = {v: k for k, v in state['class_to_idx'].items()}
print(f'モデル読み込み完了 (val_acc={state["val_acc"]:.4f})')

infer_transform = build_transforms(train=False)

class InferDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths     = paths
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        try:
            img = Image.open(p).convert('RGB')
            return self.transform(img), str(p)
        except Exception:
            return torch.zeros(3, 260, 260), str(p)

input_dir = Path(PREDICT_INPUT)
out_dirs  = {
    'farmland': Path(DATA_DIR) / 'farmland',
    'problem':  Path(DATA_DIR) / 'problem',
    'review':   Path(DATA_DIR) / 'review',
}
if not DRY_RUN:
    for d in out_dirs.values():
        d.mkdir(parents=True, exist_ok=True)

all_images = sorted(input_dir.glob('*.jpg'))
print(f'推論対象: {len(all_images)}枚')

infer_ds     = InferDataset(all_images, infer_transform)
infer_loader = DataLoader(infer_ds, batch_size=BATCH_SIZE_INFER, shuffle=False,
                          num_workers=0, pin_memory=False)

records = []
with torch.no_grad():
    for batch_imgs, batch_paths in tqdm(infer_loader, desc='推論'):
        batch_imgs = batch_imgs.to(DEVICE)
        probs_all  = F.softmax(pred_model(batch_imgs), dim=1)
        for probs, img_path in zip(probs_all, batch_paths):
            farm_prob = probs[class_to_idx['farmland']].item()
            prob_prob = probs[class_to_idx['problem']].item()
            if prob_prob >= THRESHOLD_PROB:
                pred_class, confidence, dest = 'problem', prob_prob, 'problem'
            elif farm_prob >= THRESHOLD_FARM:
                pred_class, confidence, dest = 'farmland', farm_prob, 'farmland'
            else:
                pred_class = 'farmland' if farm_prob >= prob_prob else 'problem'
                confidence = max(farm_prob, prob_prob)
                dest = 'review'
            records.append({'file': Path(img_path).name, 'pred': pred_class,
                            'confidence': round(confidence, 4),
                            'farm_prob': round(farm_prob, 4),
                            'prob_prob': round(prob_prob, 4),
                            'dest': dest})
            if not DRY_RUN:
                shutil.move(img_path, out_dirs[dest] / Path(img_path).name)

df_pred = pd.DataFrame(records)
df_pred.to_csv(Path(DATA_DIR) / 'predict_report.csv', index=False)
print('\n--- 推論結果 ---')
print(df_pred.dest.value_counts().to_string())
print(f'\n平均確信度: {df_pred.confidence.mean():.4f}')
if DRY_RUN:
    print('\n※ DRY_RUN=True のため移動していません。False にして再実行してください。')

In [ ]:
# ===== セル16: 確信度分布グラフ + review画像サムネイル =====
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_pred.confidence, bins=30, edgecolor='black')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', label=f'threshold={THRESHOLD}')
axes[0].set_xlabel('確信度'); axes[0].set_ylabel('件数')
axes[0].set_title('確信度分布'); axes[0].legend()
df_pred.dest.value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('振り分け結果'); axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

review_images = sorted(Path('data/review').glob('*.jpg'))[:16]
if review_images:
    cols = 4
    rows = (len(review_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    for ax, p in zip(axes.flat, review_images):
        row = df_pred[df_pred.file == p.name]
        conf = row.confidence.values[0] if len(row) else 0
        pred = row.pred.values[0] if len(row) else '?'
        ax.imshow(Image.open(p))
        ax.set_title(f'pred={pred}\nconf={conf:.2f}', fontsize=8)
        ax.axis('off')
    for ax in axes.flat[len(review_images):]:
        ax.axis('off')
    plt.suptitle('要確認画像（review/）→ farmland/ or problem/ に手動で移動してください')
    plt.tight_layout()
    plt.show()
else:
    print('review/ に画像がありません')

In [ ]:
# ===== セル18: problem確認HTMLビューワー（カテゴリ分類 + 大画像表示） =====
from pathlib import Path
import pandas as pd

REGION   = 'Tokyo'   # ← 都道府県名（Tokyo / Kanagawa / Niigata など）
DATA_DIR = f'/Users/nk19187/Downloads/datafarm{REGION.lower()}'
df = pd.read_csv(f'{DATA_DIR}/predict_report.csv')

def find_image(fname):
    fname = str(fname).strip()
    for sub in ['problem', 'farmland', 'unlabeled', 'review']:
        p = Path(DATA_DIR) / sub / fname
        if p.exists():
            return p
    return None

problem_all = df[df['pred']=='problem'].sort_values('confidence', ascending=False)
print(f'problem全件: {len(problem_all)}件')

cards = ''
found = 0
for _, row in problem_all.iterrows():
    p = find_image(row['file'])
    if not p:
        continue
    found += 1
    fname = row['file']
    conf  = f"{row['confidence']:.3f}"
    prefix = fname.split('_')[0] if '_' in fname else ''
    cards += (
        '<div class="card" data-fname="' + fname + '" data-cat="">'
        + '<div class="land-type">' + prefix + '</div>'
        + '<img src="file://' + str(p) + '" loading="lazy">'
        + '<div class="label"><span class="conf">' + conf + '</span><br>' + fname + '</div>'
        + '<div class="cat-buttons">'
        + '<button class="cb" data-cat="駐車場" onclick="setcat(event,this,&apos;駐車場&apos;)">🅿 駐車場</button>'
        + '<button class="cb" data-cat="ヤード系" onclick="setcat(event,this,&apos;ヤード系&apos;)">🏗 ヤード系</button>'
        + '<button class="cb" data-cat="道路" onclick="setcat(event,this,&apos;道路&apos;)">🛣 道路</button>'
        + '<button class="cb" data-cat="迷い" onclick="setcat(event,this,&apos;迷い&apos;)">🤔 迷い</button>'
        + '<button class="cb" data-cat="その他" onclick="setcat(event,this,&apos;その他&apos;)">❓ その他</button>'
        + '</div></div>'
    )

toolbar = """<div class="toolbar">
  <button id="btn-dl-p" onclick="dl('駐車場')">💾 駐車場 DL</button>
  <button id="btn-dl-y" onclick="dl('ヤード系')">💾 ヤード系 DL</button>
  <button id="btn-dl-r" onclick="dl('道路')">💾 道路 DL</button>
  <button id="btn-dl-m" onclick="dl('迷い')">💾 迷い DL</button>
  <button id="btn-dl-o" onclick="dl('その他')">💾 その他 DL</button>
  <button id="btn-clear" onclick="clearAll()">✕ 解除</button>
  <span id="count">0件分類済み</span>
  <span id="pos">0 / 0</span>
</div>"""

script = """<script>
function setcat(e, btn, cat) {
  e.stopPropagation();
  var card = btn.closest('.card');
  var prev = card.dataset.cat;
  if (prev === cat) {
    card.dataset.cat = '';
    card.querySelectorAll('.cb').forEach(function(b){ b.className='cb'; });
  } else {
    card.dataset.cat = cat;
    card.querySelectorAll('.cb').forEach(function(b){ b.className='cb'; });
    btn.className = 'cb active-' + cat;
  }
  updateCount();
}
function updateCount() {
  var n = [].slice.call(document.querySelectorAll('.card')).filter(function(c){ return c.dataset.cat; }).length;
  document.getElementById('count').textContent = n + '件分類済み';
}
function updatePos() {
  var cards = document.querySelectorAll('.card');
  var total = cards.length;
  if (!total) return;
  var mid = window.innerHeight / 2;
  var current = 0;
  for (var i = 0; i < cards.length; i++) {
    var r = cards[i].getBoundingClientRect();
    if (r.top <= mid) current = i + 1;
  }
  document.getElementById('pos').textContent = current + ' / ' + total;
}
function dl(cat) {
  var files = [].slice.call(document.querySelectorAll('.card[data-cat="' + cat + '"]'))
                  .map(function(c){ return c.dataset.fname; });
  if (!files.length) { alert(cat + ' は0件です'); return; }
  var blob = new Blob([files.join('\\n')], {type: 'text/plain'});
  var a = document.createElement('a');
  a.href = URL.createObjectURL(blob);
  a.download = cat + '_list.txt';
  a.click();
}
function clearAll() {
  document.querySelectorAll('.card').forEach(function(c){
    c.dataset.cat = '';
    c.querySelectorAll('.cb').forEach(function(b){ b.className='cb'; });
  });
  updateCount();
}
window.addEventListener('scroll', updatePos, {passive: true});
window.addEventListener('load', updatePos);
</script>"""

css = """<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: sans-serif; background: #111; color: #eee; padding: 20px; }
h1 { color: #f66; margin-bottom: 10px; }
.toolbar { position: sticky; top: 0; background: #1a1a1a; padding: 10px 12px; z-index: 100;
           border-bottom: 1px solid #444; margin-bottom: 16px;
           display: flex; flex-wrap: wrap; gap: 8px; align-items: center; }
.toolbar button { padding: 7px 12px; border: none; border-radius: 6px;
                  cursor: pointer; font-size: 13px; font-weight: bold; }
#btn-dl-p  { background: #e74; color: #fff; }
#btn-dl-y  { background: #e94; color: #fff; }
#btn-dl-r  { background: #585; color: #fff; }
#btn-dl-m  { background: #a6c; color: #fff; }
#btn-dl-o  { background: #69c; color: #fff; }
#btn-clear { background: #555; color: #fff; }
#count { color: #fa0; font-weight: bold; font-size: 13px; }
#pos   { margin-left: auto; color: #8cf; font-weight: bold; font-size: 14px; }
.grid { display: flex; flex-wrap: wrap; gap: 12px; }
.card { width: 360px; background: #222; border-radius: 8px; overflow: hidden;
        position: relative; border: 3px solid transparent; transition: border 0.1s; }
.card[data-cat="駐車場"] { border-color: #e74; }
.card[data-cat="ヤード系"] { border-color: #e94; }
.card[data-cat="道路"]    { border-color: #585; }
.card[data-cat="迷い"]    { border-color: #a6c; }
.card[data-cat="その他"]  { border-color: #69c; }
.land-type { background: #333; color: #ff9; font-size: 13px; font-weight: bold;
             padding: 4px 10px; letter-spacing: 0.05em; }
.card img { width: 360px; height: 360px; object-fit: cover; display: block; }
.label { font-size: 11px; padding: 5px 8px; word-break: break-all; color: #aaa; }
.conf { color: #f66; font-weight: bold; }
.cat-buttons { display: flex; gap: 5px; padding: 8px; background: #1a1a1a; }
.cb { flex: 1; padding: 7px 0; border: none; border-radius: 5px;
      cursor: pointer; font-size: 11px; font-weight: bold; background: #333; color: #ccc; transition: background 0.15s; }
.cb:hover { background: #555; }
.cb.active-駐車場  { background: #e74; color: #fff; }
.cb.active-ヤード系 { background: #e94; color: #fff; }
.cb.active-道路    { background: #585; color: #fff; }
.cb.active-迷い    { background: #a6c; color: #fff; }
.cb.active-その他  { background: #69c; color: #fff; }
</style>"""

html = (
    '<!DOCTYPE html>\n<html><head><meta charset="utf-8"><title>problem確認</title>\n'
    + css
    + '</head>\n<body>\n'
    + '<h1>🏗️ problem確認 — カテゴリを選んで分類</h1>\n'
    + toolbar
    + '\n<div class="grid">' + cards + '</div>\n'
    + script
    + '\n</body></html>'
)

out = DATA_DIR + '/problem_check.html'
with open(out, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'表示: {found}件')
print(f'保存: {out}')


In [ ]:
# ===== セル17: 継続学習（corrected/farmland重み付き + pseudo-label + backbone凍結） =====
import random
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image

BASE_MODEL            = 'models/model_niigatav0.pth'
NEW_MODEL_OUT         = 'models/model_niigatav1.pth'
RETRAIN_EPOCHS        = 50
RETRAIN_LR            = 2e-5
PATIENCE              = 10
CORRECTED_WEIGHT_FARM = 5   # corrected/farmland の重み倍率
PSEUDO_THRESHOLD      = 0.95
PSEUDO_MAX            = 5000

TRAIN_CLASSES = sorted(['farmland', 'problem'])
class_to_idx  = {cls: i for i, cls in enumerate(TRAIN_CLASSES)}

class SimpleImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

def _build_model_for_load(ckpt, num_classes=2):
    """B0/B2をブロック数で判別（features.1.1はB2のみ存在、B0にはない）。"""
    if 'features.1.1.block.0.0.weight' in ckpt['model']:
        m = models.efficientnet_b2(weights=None)
        print('アーキテクチャ自動判別: B2')
    else:
        m = models.efficientnet_b0(weights=None)
        print('アーキテクチャ自動判別: B0')
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

# ファイル探索
_search_dirs = [
    Path(DATA_DIR) / 'farmland',
    Path(DATA_DIR) / 'problem',
    Path(DATA_DIR) / 'unlabeled',
    Path(DATA_DIR) / 'review',
]
def _resolve_path(p):
    p = Path(p)
    if p.exists():
        return str(p.resolve())
    for d in _search_dirs:
        candidate = d / p.name
        if candidate.exists():
            return str(candidate.resolve())
    return str(p)

# ===== Phase 1: farmland_list / problem_list =====
samples = (
    [(_resolve_path(p), class_to_idx['farmland']) for p in farmland_list] +
    [(_resolve_path(p), class_to_idx['problem'])  for p in problem_list]
)
samples = [(p, lbl) for p, lbl in samples if Path(p).exists()]

# ===== Phase 2: corrected/farmland（重み付き） =====
cd_farm = Path(DATA_DIR) / 'corrected' / 'farmland'
corrected_f = 0
if cd_farm.exists():
    imgs = list(cd_farm.glob('*.jpg'))
    corrected_f = len(imgs)
    for p in imgs:
        for _ in range(CORRECTED_WEIGHT_FARM):
            samples.append((str(p), class_to_idx['farmland']))

# ===== Phase 3: pseudo-label（高確信度farmland） =====
report_path = Path(DATA_DIR) / 'predict_report.csv'
pseudo_count = 0
if report_path.exists():
    df_report = pd.read_csv(report_path)
    prob_col = 'farm_prob' if 'farm_prob' in df_report.columns else 'confidence'
    pseudo_df = df_report[
        (df_report.dest == 'farmland') & (df_report[prob_col] >= PSEUDO_THRESHOLD)
    ].head(PSEUDO_MAX)
    for _, row in pseudo_df.iterrows():
        p = _resolve_path(row['file'])
        if Path(p).exists():
            samples.append((p, class_to_idx['farmland']))
            pseudo_count += 1
    print(f'pseudo-label farmland: {pseudo_count}枚追加（{prob_col}>={PSEUDO_THRESHOLD}）')

# ===== 集計 =====
counts = [0, 0]
for _, lbl in samples:
    counts[lbl] += 1

print(f'学習サンプル数: {len(samples)}')
for cls in TRAIN_CLASSES:
    print(f'  {cls}: {counts[class_to_idx[cls]]}枚')
print(f'  (corrected/farmland={corrected_f}枚 x{CORRECTED_WEIGHT_FARM}倍, pseudo={pseudo_count}枚)')

if 0 in counts:
    missing_cls = [TRAIN_CLASSES[i] for i, c in enumerate(counts) if c == 0]
    raise ValueError(f'⚠️ {missing_cls} に画像がありません')

# ===== データローダー =====
random.seed(42)
random.shuffle(samples)
n_val   = max(1, int(len(samples) * VAL_RATIO))
n_train = len(samples) - n_val
train_ds = SimpleImageDataset(samples[:n_train], transform=build_transforms(train=True))
val_ds   = SimpleImageDataset(samples[n_train:], transform=build_transforms(train=False))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f'train={n_train}枚, val={n_val}枚')

# ===== モデル読み込み（アーキテクチャ自動判別） =====
retrain_state = torch.load(BASE_MODEL, map_location=DEVICE)
retrain_model = _build_model_for_load(retrain_state, num_classes=2).to(DEVICE)
retrain_model.load_state_dict(retrain_state['model'])
print(f'起点モデル val_acc={retrain_state["val_acc"]:.4f}')

# ===== backbone凍結（最終2ブロック + classifier のみ学習） =====
for param in retrain_model.parameters():
    param.requires_grad = False
for block in list(retrain_model.features)[-2:]:
    for param in block.parameters():
        param.requires_grad = True
for param in retrain_model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in retrain_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in retrain_model.parameters())
print(f'学習パラメータ: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# ===== 学習 =====
class_weights = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, retrain_model.parameters()),
    lr=RETRAIN_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=RETRAIN_EPOCHS)

best_r, no_improve = 0.0, 0
Path(NEW_MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(1, RETRAIN_EPOCHS + 1), desc='継続学習'):
    _, train_acc_r = run_epoch(retrain_model, train_loader, criterion, optimizer)
    _, val_acc_r   = run_epoch(retrain_model, val_loader,   criterion)
    scheduler.step()
    tqdm.write(f'Epoch {epoch:03d} | train={train_acc_r:.4f} | val={val_acc_r:.4f}')
    if val_acc_r >= best_r:
        best_r     = val_acc_r
        no_improve = 0
        torch.save({'epoch': epoch, 'model': retrain_model.state_dict(),
                    'class_to_idx': class_to_idx, 'val_acc': val_acc_r}, NEW_MODEL_OUT)
        tqdm.write(f'  → 保存: {NEW_MODEL_OUT} (val_acc={val_acc_r:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            tqdm.write(f'Early stopping (patience={PATIENCE})')
            break

print(f'\n継続学習完了: {retrain_state["val_acc"]:.4f} → {best_r:.4f}')
print(f'次回: PREDICT_MODEL と BASE_MODEL を {NEW_MODEL_OUT} に変更して再実行')